# WHAS500 Patient Survival Prediction Calculator

> **Experimental Research Tool** — This calculator is built for machine learning portfolio
> demonstration and educational purposes. It is **not** a clinically validated medical device
> and must not be used for actual medical decision-making.

**Dataset:** WHAS500 — Worcester Heart Attack Study (500 patients)
**Models:** Cox Proportional Hazards, Weibull Accelerated Failure Time
**Interface:** Gradio web application
**Prediction Baseline:** Hospital admission for acute myocardial infarction

---

## 1. Environment Setup

In [2]:
import subprocess, sys

for pkg in ['gradio', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import joblib
import os

from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullAFTFitter
from lifelines.utils import concordance_index as lifelines_ci
from sklearn.model_selection import KFold

import gradio as gr

print("All libraries loaded successfully.")
print(f"Gradio version: {gr.__version__}")

C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded successfully.
Gradio version: 6.30.0


## 2. Notebook Audit & Model Selection

### 2.1 Audit of WHAS500_Survival_Analysis.ipynb

| Item | Finding |
|------|---------|
| **Dataset** | WHAS500 — 500 patients hospitalized for acute myocardial infarction |
| **Event definition** | Death (`event` = 1), right-censored otherwise |
| **Time variable** | `time` — follow-up in days (range: 1 to 2,358) |
| **Censoring rate** | 57% censored (survived), 43% died (215 deaths) |
| **Predictors** | 14 clinical features: age, bmi, hr, sysbp, diasbp, los, afb, av3, chf, cvd, gender, miord, mitype, sho |
| **Models trained** | Cox PH (penalizer=0.01), Weibull AFT (penalizer=0.01), RSF (IPCW + RandomForestRegressor, 200 trees) |
| **Evaluation** | 5-fold cross-validated concordance index, Integrated Brier Score (IPCW) |
| **Missing values** | None — dataset is complete |
| **Encoding** | Binary features coded as 0/1; no additional encoding or scaling applied |

### 2.2 Data Leakage Assessment

**`los` (length of hospital stay)** is excluded from the calculator. This variable is recorded at discharge and is unavailable at hospital admission, which is the intended prediction baseline. Including it would introduce look-ahead bias.

All remaining features (demographics, vital signs at admission, cardiac history) are available at or before admission and are safe for prediction.

### 2.3 Recommended Model

**Primary: Cox Proportional Hazards** — selected for its ability to produce all required outputs with minimal assumptions.

| Criterion | Cox PH | Weibull AFT | RSF (IPCW) |
|-----------|--------|-------------|------------|
| Full survival curve | Yes | Yes | No (time only) |
| Hazard-based risk score | Yes | No | No |
| Median survival estimate | Yes | Yes | No |
| Interpretability | High (hazard ratios) | High (acceleration factors) | Low |
| Distributional assumptions | None (semi-parametric) | Weibull (parametric) | None |
| Clinical acceptance | Standard of care | Established | Emerging |

Weibull AFT is included as a secondary option for comparison. RSF is excluded because the IPCW-based implementation only predicts survival time — it cannot generate the survival probability curves required by the dashboard.

## 3. Data Loading & Feature Preparation

In [3]:
df = pd.read_csv('whas500.csv')

features = [c for c in df.columns if c not in ('time', 'event', 'los')]

continuous = ['age', 'bmi', 'diasbp', 'hr', 'sysbp']
binary = ['afb', 'av3', 'chf', 'cvd', 'gender', 'miord', 'mitype', 'sho']

print(f"Dataset: {df.shape[0]} patients, {len(features)} predictor features")
print(f"Events: {df['event'].sum()} deaths ({df['event'].mean()*100:.1f}%)")
print(f"Follow-up: {df['time'].min()} to {df['time'].max()} days (median {df['time'].median():.0f})")
print(f"\nFeatures ({len(features)}): {features}")
print(f"Excluded: los (unavailable at prediction baseline)")
print(f"Missing values: {df[features].isnull().sum().sum()}")

Dataset: 500 patients, 13 predictor features
Events: 215 deaths (43.0%)
Follow-up: 1.0 to 2358.0 days (median 632)

Features (13): ['afb', 'age', 'av3', 'bmi', 'chf', 'cvd', 'diasbp', 'gender', 'hr', 'miord', 'mitype', 'sho', 'sysbp']
Excluded: los (unavailable at prediction baseline)
Missing values: 0


## 4. Model Training

Models are retrained on 13 features (excluding `los`) using the same hyperparameters as the analysis notebook. This ensures predictions reflect only information available at admission.

### 4.1 Cox Proportional Hazards

In [4]:
cph = CoxPHFitter(penalizer=0.01)
cph.fit(df[features + ['time', 'event']], duration_col='time', event_col='event')

print(f"Cox PH trained successfully")
print(f"In-sample C-index: {cph.concordance_index_:.4f}")
cph.print_summary(columns=['coef', 'exp(coef)', 'p'])

Cox PH trained successfully
In-sample C-index: 0.7838


<lifelines.CoxPHFitter: fitted with 500 total observations, 285 right-censored observations>
             duration col = 'time'
                event col = 'event'
                penalizer = 0.01
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 500
number of events observed = 215
   partial log-likelihood = -1116.91
         time fit was run = 2026-10-13 18:51:25 UTC

---
           coef exp(coef)      p
covariate                       
afb        0.02      1.02   0.90
age        0.05      1.05 <0.005
av3        0.32      1.38   0.45
bmi       -0.05      0.95 <0.005
chf        0.72      2.06 <0.005
cvd       -0.05      0.95   0.76
diasbp    -0.01      0.99   0.01
gender    -0.27      0.76   0.05
hr         0.01      1.01 <0.005
miord      0.09      1.09   0.56
mitype    -0.25      0.78   0.17
sho        1.19      3.29 <0.005
sysbp      0.00      1.00   0.65
---
Concordance = 0.78
Partial AIC = 2259.83
log-likelihood ratio test = 220.81 on 13 df
-log2(p) of ll-ratio test = 130.05

### 4.2 Weibull AFT

In [5]:
waft = WeibullAFTFitter(penalizer=0.01)
waft.fit(df[features + ['time', 'event']], duration_col='time', event_col='event')

print(f"Weibull AFT trained successfully")
print(f"In-sample C-index: {waft.concordance_index_:.4f}")
waft.print_summary(columns=['coef', 'exp(coef)', 'p'])

Weibull AFT trained successfully
In-sample C-index: 0.7842


<lifelines.WeibullAFTFitter: fitted with 500 total observations, 285 right-censored observations>
             duration col = 'time'
                event col = 'event'
                penalizer = 0.01
   number of observations = 500
number of events observed = 215
           log-likelihood = -1647.21
         time fit was run = 2026-10-13 18:51:34 UTC

---
                   coef exp(coef)      p
param   covariate                       
lambda_ afb       -0.02      0.98   0.96
        age       -0.07      0.93 <0.005
        av3       -0.83      0.44   0.24
        bmi        0.08      1.08 <0.005
        chf       -1.27      0.28 <0.005
        cvd        0.08      1.08   0.79
        diasbp     0.02      1.02   0.01
        gender     0.39      1.48   0.10
        hr        -0.02      0.98 <0.005
        miord     -0.13      0.88   0.61
        mitype     0.45      1.57   0.14
        sho       -2.14      0.12 <0.005
        sysbp     -0.00      1.00   0.87
        Intercept 11.96  1.57e+05 <0.005
rho_    Intercept -0.55      0.57 <0.005
---
Concordance = 0.78
AIC = 3324.42
log-likelihood ratio test = 210.48 on 13 df
-log2(p) of ll-ratio test = 122.98

### 4.3 Cross-Validated Performance

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cox_scores, aft_scores = [], []

for fold, (train_idx, test_idx) in enumerate(kf.split(df), 1):
    train, test = df.iloc[train_idx], df.iloc[test_idx]

    cph_fold = CoxPHFitter(penalizer=0.01)
    cph_fold.fit(train[features + ['time', 'event']], duration_col='time', event_col='event')
    preds = cph_fold.predict_partial_hazard(test[features]).squeeze()
    cox_scores.append(lifelines_ci(test['time'], -preds.values, test['event']))

    aft_fold = WeibullAFTFitter(penalizer=0.01)
    aft_fold.fit(train[features + ['time', 'event']], duration_col='time', event_col='event')
    preds = aft_fold.predict_median(test[features]).squeeze()
    aft_scores.append(lifelines_ci(test['time'], preds.values, test['event']))

print("5-Fold Cross-Validated Concordance Index (without los)")
print(f"  Cox PH:      {np.mean(cox_scores):.4f} +/- {np.std(cox_scores):.4f}")
print(f"  Weibull AFT: {np.mean(aft_scores):.4f} +/- {np.std(aft_scores):.4f}")
print(f"\nNote: C-index > 0.5 indicates better-than-random discrimination.")
print(f"These scores exclude los and may differ from the analysis notebook.")

5-Fold Cross-Validated Concordance Index (without los)
  Cox PH:      0.7664 +/- 0.0359
  Weibull AFT: 0.7659 +/- 0.0384

Note: C-index > 0.5 indicates better-than-random discrimination.
These scores exclude los and may differ from the analysis notebook.


## 5. Model Persistence

Trained models are saved to disk so the calculator can be reloaded without retraining.

In [7]:
os.makedirs('models', exist_ok=True)

joblib.dump(cph, 'models/cox_model.pkl')
joblib.dump(waft, 'models/aft_model.pkl')
joblib.dump(features, 'models/feature_list.pkl')

print("Models saved to ./models/")
for f in sorted(os.listdir('models')):
    size_kb = os.path.getsize(f'models/{f}') / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

print("\nTo reload without retraining:")
print("  cph  = joblib.load('models/cox_model.pkl')")
print("  waft = joblib.load('models/aft_model.pkl')")

Models saved to ./models/
  aft_model.pkl (43.3 KB)
  cox_model.pkl (86.5 KB)
  feature_list.pkl (0.1 KB)

To reload without retraining:
  cph  = joblib.load('models/cox_model.pkl')
  waft = joblib.load('models/aft_model.pkl')


## 6. Prediction Engine

The prediction engine validates patient inputs, generates survival probabilities at five
clinically relevant time horizons, computes median survival time and relative risk scores,
and renders the results as an interactive dashboard.

In [8]:
HORIZONS = [30, 90, 365, 1095, 1825]
HORIZON_LABELS = ['30 Days', '90 Days', '1 Year', '3 Years', '5 Years']
MAX_FOLLOW_UP = int(df['time'].max())

VALIDATION_RANGES = {
    'age': (20, 110, 'years'),
    'bmi': (10, 60, 'kg/m\u00b2'),
    'hr': (30, 250, 'bpm'),
    'sysbp': (50, 300, 'mmHg'),
    'diasbp': (20, 200, 'mmHg'),
}


def validate_inputs(age, bmi, hr, sysbp, diasbp):
    errors = []
    for name, val, key in [('Age', age, 'age'), ('BMI', bmi, 'bmi'),
                            ('Heart Rate', hr, 'hr'), ('Systolic BP', sysbp, 'sysbp'),
                            ('Diastolic BP', diasbp, 'diasbp')]:
        lo, hi, unit = VALIDATION_RANGES[key]
        if val is None or np.isnan(val):
            errors.append(f"{name} is required")
        elif not (lo <= val <= hi):
            errors.append(f"{name} must be between {lo} and {hi} {unit}")
    if age is not None and bmi is not None and hr is not None:
        if sysbp is not None and diasbp is not None and diasbp >= sysbp:
            errors.append("Diastolic BP must be lower than Systolic BP")
    return errors


def build_patient_df(age, bmi, hr, sysbp, diasbp,
                     gender, chf, afb, cvd, sho, miord, mitype, av3):
    patient = pd.DataFrame([{
        'afb': 1 if afb == "Yes" else 0,
        'age': float(age),
        'av3': 1 if av3 == "Yes" else 0,
        'bmi': float(bmi),
        'chf': 1 if chf == "Yes" else 0,
        'cvd': 1 if cvd == "Yes" else 0,
        'diasbp': float(diasbp),
        'gender': 1 if gender == "Female" else 0,
        'hr': float(hr),
        'miord': 1 if miord == "Recurrent" else 0,
        'mitype': 1 if mitype == "Q-wave" else 0,
        'sho': 1 if sho == "Yes" else 0,
        'sysbp': float(sysbp),
    }])
    return patient[features]


def compute_predictions(patient_df, model_name):
    model = cph if model_name == "Cox PH" else waft

    curve_times = np.linspace(1, MAX_FOLLOW_UP, 500)
    surv = model.predict_survival_function(patient_df, times=curve_times)
    surv_probs = surv.values.ravel()

    curve_times = np.concatenate([[0], curve_times])
    surv_probs = np.concatenate([[1.0], surv_probs])

    horizon_probs = {}
    for t, label in zip(HORIZONS, HORIZON_LABELS):
        if t <= MAX_FOLLOW_UP:
            s = model.predict_survival_function(patient_df, times=[t])
            horizon_probs[label] = float(np.clip(s.values.ravel()[0], 0, 1))

    median_surv = float(model.predict_median(patient_df).squeeze())

    risk_score = None
    risk_percentile = None
    if model_name == "Cox PH":
        risk_score = float(cph.predict_partial_hazard(patient_df).squeeze())
        all_risks = cph.predict_partial_hazard(df[features]).squeeze().values
        risk_percentile = float((all_risks <= risk_score).mean() * 100)

    return {
        'times': curve_times,
        'survival': surv_probs,
        'horizon_probs': horizon_probs,
        'median_survival': median_surv,
        'risk_score': risk_score,
        'risk_percentile': risk_percentile,
        'model': model_name,
    }

print("Prediction engine defined.")

Prediction engine defined.


### 6.1 Dashboard Rendering

In [9]:
def prob_color(p):
    if p >= 0.80:
        return '#198754'
    elif p >= 0.50:
        return '#fd7e14'
    elif p >= 0.20:
        return '#dc3545'
    return '#6f42c1'


def build_summary_html(inputs_dict, predictions):
    model = predictions['model']
    rows = ""
    for key, val in inputs_dict.items():
        rows += f'<div><span style="color:#6c757d;">{key}:</span> <strong>{val}</strong></div>\n'

    median = predictions['median_survival']
    if np.isinf(median) or median > MAX_FOLLOW_UP:
        median_text = f"Not reached within follow-up period (>{MAX_FOLLOW_UP} days)"
    else:
        median_text = f"{median:,.0f} days ({median/365.25:.1f} years)"

    risk_html = ""
    if predictions['risk_score'] is not None:
        risk_html = f'''
        <div style="background:#fff8f0; border:1px solid #ffd699; border-radius:8px;
                    padding:16px; text-align:center;">
            <div style="font-size:11px; color:#cc7a00; text-transform:uppercase;
                        letter-spacing:0.5px;">Relative Risk Score</div>
            <div style="font-size:24px; font-weight:700; color:#cc7a00; margin:4px 0;">
                {predictions["risk_score"]:.3f}</div>
            <div style="font-size:12px; color:#997a4d;">
                {predictions["risk_percentile"]:.0f}th percentile in cohort</div>
        </div>'''

    return f'''
    <div style="font-family:'Segoe UI',Tahoma,sans-serif;">
        <div style="background:#f8f9fa; border:1px solid #dee2e6; border-radius:8px;
                    padding:20px; margin-bottom:16px;">
            <div style="display:flex; justify-content:space-between; align-items:center;
                        margin-bottom:12px;">
                <h3 style="margin:0; color:#2c3e50; font-size:16px;">Patient Summary</h3>
                <span style="background:#198754; color:white; padding:4px 12px;
                             border-radius:20px; font-size:11px;">Valid Input</span>
            </div>
            <div style="display:grid; grid-template-columns:repeat(3, 1fr);
                        gap:6px; font-size:13px;">{rows}</div>
            <div style="margin-top:10px; font-size:12px; color:#6c757d;">
                Model: <strong>{model}</strong> &nbsp;|&nbsp;
                Baseline: Hospital admission &nbsp;|&nbsp;
                Time units: Days
            </div>
        </div>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-bottom:16px;">
            <div style="background:#f0f7ff; border:1px solid #b3d7ff; border-radius:8px;
                        padding:16px; text-align:center;">
                <div style="font-size:11px; color:#0056b3; text-transform:uppercase;
                            letter-spacing:0.5px;">Median Survival Time</div>
                <div style="font-size:20px; font-weight:700; color:#0056b3; margin:4px 0;">
                    {median_text}</div>
            </div>
            {risk_html}
        </div>
    </div>'''


def build_cards_html(predictions):
    cards = ""
    for label, prob in predictions['horizon_probs'].items():
        color = prob_color(prob)
        mort = 1.0 - prob
        cards += f'''
        <div style="background:white; border:1px solid #e9ecef; border-radius:8px;
                    padding:16px; text-align:center; border-top:3px solid {color};">
            <div style="font-size:11px; color:#6c757d; text-transform:uppercase;
                        letter-spacing:0.5px;">{label}</div>
            <div style="font-size:28px; font-weight:700; color:{color};
                        margin:8px 0;">{prob*100:.1f}%</div>
            <div style="font-size:11px; color:#6c757d;">Survival Probability</div>
            <hr style="margin:8px 0; border:none; border-top:1px solid #f0f0f0;">
            <div style="font-size:11px; color:#dc3545;">{mort*100:.1f}% Estimated Mortality</div>
        </div>'''

    return f'''
    <div style="font-family:'Segoe UI',Tahoma,sans-serif;">
        <div style="display:grid; grid-template-columns:repeat(5, 1fr);
                    gap:10px; margin-bottom:16px;">{cards}</div>
    </div>'''


def build_survival_plot(predictions):
    times = predictions['times']
    survival = predictions['survival'] * 100

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=times, y=survival,
        mode='lines',
        name='Survival Probability',
        line=dict(color='#1a73e8', width=3),
        fill='tozeroy',
        fillcolor='rgba(26, 115, 232, 0.08)',
        hovertemplate='Day %{x:.0f}<br>Survival: %{y:.1f}%<extra></extra>'
    ))

    for t, label in zip(HORIZONS, HORIZON_LABELS):
        if label in predictions['horizon_probs']:
            p = predictions['horizon_probs'][label] * 100
            color = prob_color(predictions['horizon_probs'][label])
            fig.add_trace(go.Scatter(
                x=[t], y=[p],
                mode='markers+text',
                text=[f'{p:.1f}%'],
                textposition='top center',
                textfont=dict(size=11, color=color),
                marker=dict(size=10, color=color, line=dict(width=2, color='white')),
                showlegend=False,
                hovertemplate=f'{label}<br>Survival: {p:.1f}%<extra></extra>'
            ))

    median = predictions['median_survival']
    if not np.isinf(median) and median <= MAX_FOLLOW_UP:
        fig.add_hline(y=50, line_dash="dot", line_color="#aaa", line_width=1)
        fig.add_vline(x=median, line_dash="dot", line_color="#aaa", line_width=1)
        fig.add_annotation(x=median, y=52, text=f"Median: {median:.0f} days",
                           showarrow=False, font=dict(size=10, color="#666"))

    fig.update_layout(
        title=dict(text=f'Predicted Survival Curve ({predictions["model"]})',
                   font=dict(size=16)),
        xaxis_title='Time Since Admission (days)',
        yaxis_title='Survival Probability (%)',
        yaxis_range=[0, 105],
        xaxis_range=[0, MAX_FOLLOW_UP + 50],
        template='plotly_white',
        height=420,
        margin=dict(l=60, r=30, t=50, b=60),
        hovermode='closest',
        showlegend=False,
        font=dict(family="Segoe UI, Tahoma, sans-serif"),
    )

    return fig


def build_interpretation_html(predictions):
    model = predictions['model']
    probs = predictions['horizon_probs']

    p_1y = probs.get('1 Year', None)
    p_30d = probs.get('30 Days', None)

    if p_1y is not None and p_1y >= 0.75:
        outlook = "relatively favorable"
    elif p_1y is not None and p_1y >= 0.50:
        outlook = "moderate"
    else:
        outlook = "elevated risk"

    trend_parts = []
    prev = None
    for label in HORIZON_LABELS:
        if label in probs:
            p = probs[label]
            if prev is not None:
                drop = (prev - p) * 100
                if drop > 10:
                    trend_parts.append(f"a notable decline between the preceding horizon and {label}")
            prev = p

    trend_text = ("The survival curve shows " + " and ".join(trend_parts) + "."
                  if trend_parts else
                  "The survival probability declines gradually over the follow-up period.")

    median = predictions['median_survival']
    if np.isinf(median) or median > MAX_FOLLOW_UP:
        median_text = ("The predicted median survival time exceeds the study's follow-up period, "
                       "meaning the model estimates that this patient has a greater than 50% chance "
                       "of surviving beyond the observed data range. Extrapolation beyond this "
                       "range is not supported.")
    else:
        median_text = (f"The estimated median survival time is {median:,.0f} days "
                       f"({median/365.25:.1f} years), meaning the model predicts a 50% probability "
                       f"of surviving beyond this point.")

    risk_text = ""
    if predictions['risk_score'] is not None:
        pct = predictions['risk_percentile']
        risk_text = (f"The relative risk score places this patient at the "
                     f"{pct:.0f}th percentile of the training cohort, where "
                     f"higher percentiles indicate higher estimated hazard. ")

    return f'''
    <div style="font-family:'Segoe UI',Tahoma,sans-serif; background:#f8f9fa;
                border:1px solid #dee2e6; border-radius:8px; padding:20px;">
        <h4 style="color:#2c3e50; margin:0 0 12px 0; font-size:15px;">Clinical Interpretation</h4>
        <p style="font-size:13px; color:#333; line-height:1.6;">
            Based on the <strong>{model}</strong> model, this patient profile shows
            {outlook} one-year survival. The estimated one-year survival probability is
            <strong>{p_1y*100:.1f}%</strong>, with a corresponding estimated mortality
            of {(1-p_1y)*100:.1f}%.
        </p>
        <p style="font-size:13px; color:#333; line-height:1.6;">{trend_text}</p>
        <p style="font-size:13px; color:#333; line-height:1.6;">{median_text}</p>
        <p style="font-size:13px; color:#333; line-height:1.6;">{risk_text}</p>
        <div style="background:#fff3cd; border:1px solid #ffc107; border-radius:6px;
                    padding:12px; margin-top:12px;">
            <p style="font-size:12px; color:#664d03; margin:0; line-height:1.5;">
                <strong>Important Limitations:</strong> These are population-level estimates
                derived from a statistical model trained on 500 patients from a single center.
                Individual outcomes depend on many clinical factors beyond those captured here,
                including treatment decisions, post-discharge care, subsequent cardiac events,
                and comorbidities not recorded in the dataset. Survival probabilities represent
                the model's best estimate for patients with similar characteristics — they are
                not certainties. This tool does not provide diagnoses, treatment recommendations,
                or prognoses for individual patients. Always consult a qualified healthcare
                provider for clinical decision-making.
            </p>
        </div>
    </div>'''

print("Dashboard rendering functions defined.")

Dashboard rendering functions defined.


### 6.2 Prediction Callback

In [10]:
def predict_callback(age, bmi, hr, sysbp, diasbp,
                     gender, chf, afb, cvd, sho, miord, mitype, av3,
                     model_choice):

    errors = validate_inputs(age, bmi, hr, sysbp, diasbp)
    if errors:
        error_html = '<div style="font-family:Segoe UI,sans-serif;">'
        for e in errors:
            error_html += f'<p style="color:#dc3545; margin:4px 0;">&#9888; {e}</p>'
        error_html += '</div>'
        return error_html, "", None, ""

    patient_df = build_patient_df(age, bmi, hr, sysbp, diasbp,
                                   gender, chf, afb, cvd, sho, miord, mitype, av3)
    predictions = compute_predictions(patient_df, model_choice)

    inputs_dict = {
        'Age': f'{age:.0f} years', 'BMI': f'{bmi:.1f} kg/m\u00b2',
        'Heart Rate': f'{hr:.0f} bpm', 'Systolic BP': f'{sysbp:.0f} mmHg',
        'Diastolic BP': f'{diasbp:.0f} mmHg', 'Gender': gender,
        'CHF': chf, 'Atrial Fibrillation': afb, 'CVD History': cvd,
        'Cardiogenic Shock': sho, 'MI Order': miord, 'MI Type': mitype,
        'Complete Heart Block': av3,
    }

    summary = build_summary_html(inputs_dict, predictions)
    cards = build_cards_html(predictions)
    plot = build_survival_plot(predictions)
    interp = build_interpretation_html(predictions)

    return summary, cards, plot, interp


def reset_callback():
    defaults = [65, 26.0, 80, 130, 80,
                "Male", "No", "No", "No", "No", "First", "Non-Q-wave", "No",
                "Cox PH"]
    empty = ["", "", None, ""]
    return defaults + empty

print("Callbacks defined.")

Callbacks defined.


## 7. Validation Suite

Automated tests verify that predictions are consistent, probabilities satisfy mathematical
constraints, and invalid inputs are handled correctly.

In [11]:
test_profiles = [
    {"label": "Low-risk patient",
     "args": (45, 25.0, 72, 120, 78, "Male", "No", "No", "No", "No", "First", "Non-Q-wave", "No")},
    {"label": "High-risk patient",
     "args": (88, 30.0, 110, 90, 55, "Female", "Yes", "Yes", "Yes", "Yes", "Recurrent", "Q-wave", "Yes")},
    {"label": "Average patient",
     "args": (70, 27.0, 85, 135, 80, "Male", "No", "No", "Yes", "No", "First", "Q-wave", "No")},
]

all_passed = True
print("=" * 60)
print("VALIDATION SUITE")
print("=" * 60)

for profile in test_profiles:
    print(f"\nProfile: {profile['label']}")
    patient_df = build_patient_df(*profile['args'])

    for model_name in ["Cox PH", "Weibull AFT"]:
        preds = compute_predictions(patient_df, model_name)

        probs = list(preds['horizon_probs'].values())
        in_range = all(0 <= p <= 1 for p in probs)
        non_increasing = all(probs[i] >= probs[i+1] for i in range(len(probs)-1))
        mortality_ok = all(abs((1 - p) + p - 1.0) < 1e-10 for p in probs)
        curve_starts_1 = abs(preds['survival'][0] - 1.0) < 1e-10
        curve_non_neg = all(s >= 0 for s in preds['survival'])

        status = "PASS" if all([in_range, non_increasing, mortality_ok,
                                 curve_starts_1, curve_non_neg]) else "FAIL"
        if status == "FAIL":
            all_passed = False

        print(f"  {model_name:15s} [{status}]  "
              f"30d={probs[0]*100:.1f}%  1y={probs[2]*100:.1f}%  5y={probs[4]*100:.1f}%")
        if not in_range:
            print("    FAIL: probabilities outside [0, 1]")
        if not non_increasing:
            print("    FAIL: survival not non-increasing")
        if not curve_starts_1:
            print("    FAIL: curve does not start at 1.0")

print(f"\n{'=' * 60}")
print("INVALID INPUT TESTS")
print("=" * 60)

invalid_cases = [
    ("Age out of range", (150, 25, 80, 130, 80, "Male", "No", "No", "No", "No", "First", "Non-Q-wave", "No", "Cox PH")),
    ("Diastolic >= Systolic", (65, 25, 80, 100, 110, "Male", "No", "No", "No", "No", "First", "Non-Q-wave", "No", "Cox PH")),
    ("Missing value", (None, 25, 80, 130, 80, "Male", "No", "No", "No", "No", "First", "Non-Q-wave", "No", "Cox PH")),
]

for case_name, args in invalid_cases:
    result = predict_callback(*args)
    has_error = "&#9888;" in str(result[0])
    status = "PASS" if has_error else "FAIL"
    if status == "FAIL":
        all_passed = False
    print(f"  {case_name:30s} [{status}] — error message returned: {has_error}")

print(f"\n{'=' * 60}")
if all_passed:
    print("ALL TESTS PASSED")
else:
    print("SOME TESTS FAILED — review output above")
print("=" * 60)

VALIDATION SUITE

Profile: Low-risk patient
  Cox PH          [PASS]  30d=98.2%  1y=94.6%  5y=85.8%
  Weibull AFT     [PASS]  30d=98.4%  1y=93.6%  5y=84.5%

Profile: High-risk patient
  Cox PH          [PASS]  30d=30.4%  1y=2.5%  5y=0.0%
  Weibull AFT     [PASS]  30d=33.1%  1y=1.0%  5y=0.0%

Profile: Average patient
  Cox PH          [PASS]  30d=95.7%  1y=87.3%  5y=68.6%
  Weibull AFT     [PASS]  30d=96.6%  1y=86.4%  5y=69.1%

INVALID INPUT TESTS
  Age out of range               [PASS] — error message returned: True
  Diastolic >= Systolic          [PASS] — error message returned: True
  Missing value                  [PASS] — error message returned: True

ALL TESTS PASSED


## 8. Interactive Calculator

Run the cell below to launch the Gradio web interface. It will open in your browser
at a local URL (typically `http://127.0.0.1:7860`).

Enter patient clinical features on the left and click **Predict** to generate
survival predictions on the right.

In [12]:
custom_css = '''
.gradio-container {font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;}
.gr-button-primary {background: #1a73e8 !important; border: none !important;}
.disclaimer-banner {
    background: #fff3cd; border: 1px solid #ffc107; border-radius: 8px;
    padding: 12px 16px; margin-bottom: 16px; font-size: 13px; color: #664d03;
}
'''

with gr.Blocks(css=custom_css, title="WHAS500 Survival Calculator") as demo:

    gr.HTML('''
    <div style="text-align:center; padding:16px 0 8px 0;">
        <h1 style="margin:0; color:#2c3e50; font-size:24px;">
            WHAS500 Patient Survival Prediction Calculator</h1>
        <p style="color:#6c757d; margin:4px 0 0 0; font-size:13px;">
            Worcester Heart Attack Study &mdash; Interactive Survival Analysis Tool</p>
    </div>
    <div class="disclaimer-banner">
        <strong>Experimental Research Tool</strong> &mdash; For portfolio demonstration and
        educational purposes only. Not a clinically validated medical device.
        Do not use for actual medical decisions.
    </div>
    ''')

    with gr.Row():

        with gr.Column(scale=1):
            gr.Markdown("### Patient Information")

            gr.Markdown("**Demographics & Vitals**")
            age_input = gr.Number(label="Age (years)", value=65, minimum=20, maximum=110)
            gender_input = gr.Radio(choices=["Male", "Female"], value="Male", label="Gender")
            bmi_input = gr.Number(label="BMI (kg/m\u00b2)", value=26.0, minimum=10, maximum=60)
            hr_input = gr.Number(label="Heart Rate (bpm)", value=80, minimum=30, maximum=250)
            sysbp_input = gr.Number(label="Systolic BP (mmHg)", value=130, minimum=50, maximum=300)
            diasbp_input = gr.Number(label="Diastolic BP (mmHg)", value=80, minimum=20, maximum=200)

            gr.Markdown("**Cardiac History**")
            chf_input = gr.Radio(choices=["No", "Yes"], value="No",
                                  label="Congestive Heart Failure (CHF)")
            afb_input = gr.Radio(choices=["No", "Yes"], value="No",
                                  label="Atrial Fibrillation")
            cvd_input = gr.Radio(choices=["No", "Yes"], value="No",
                                  label="Cardiovascular Disease History")
            sho_input = gr.Radio(choices=["No", "Yes"], value="No",
                                  label="Cardiogenic Shock")
            miord_input = gr.Radio(choices=["First", "Recurrent"], value="First",
                                    label="MI Order")
            mitype_input = gr.Radio(choices=["Non-Q-wave", "Q-wave"], value="Non-Q-wave",
                                     label="MI Type")
            av3_input = gr.Radio(choices=["No", "Yes"], value="No",
                                  label="Complete Heart Block (AV3)")

            gr.Markdown("**Model**")
            model_input = gr.Dropdown(choices=["Cox PH", "Weibull AFT"], value="Cox PH",
                                       label="Prediction Model")

            with gr.Row():
                predict_btn = gr.Button("Predict", variant="primary", scale=2)
                reset_btn = gr.Button("Reset", scale=1)

        with gr.Column(scale=2):
            gr.Markdown("### Results Dashboard")
            summary_output = gr.HTML()
            cards_output = gr.HTML()
            curve_output = gr.Plot()
            interp_output = gr.HTML()

    all_inputs = [age_input, bmi_input, hr_input, sysbp_input, diasbp_input,
                  gender_input, chf_input, afb_input, cvd_input, sho_input,
                  miord_input, mitype_input, av3_input, model_input]

    all_outputs = [summary_output, cards_output, curve_output, interp_output]

    predict_btn.click(fn=predict_callback, inputs=all_inputs, outputs=all_outputs)
    reset_btn.click(fn=reset_callback, inputs=[], outputs=all_inputs + all_outputs)

print("Calculator interface built. Run the next cell to launch.")

Calculator interface built. Run the next cell to launch.


### Launch the Calculator

In [13]:
demo.launch(share=False, show_error=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 9. Running Instructions

### Prerequisites

Install the required packages (no C++ compiler needed):

```
pip install lifelines scikit-learn matplotlib pandas numpy scipy gradio plotly joblib
```

### How to Run

1. Place `whas500.csv` and this notebook in the same folder.
2. Open this notebook in Jupyter Notebook or JupyterLab.
3. Run all cells in order (**Cell > Run All**).
4. The calculator launches in the last cell and opens at `http://127.0.0.1:7860`.
5. Enter patient features, select a model, and click **Predict**.

### Reloading Saved Models

After the first run, models are saved in the `./models/` folder. To skip retraining:

```python
import joblib
cph = joblib.load('models/cox_model.pkl')
waft = joblib.load('models/aft_model.pkl')
features = joblib.load('models/feature_list.pkl')
```

Then run the prediction engine and Gradio cells.

## 10. Limitations

### Model Limitations

- **Single-center study:** WHAS500 includes 500 patients from Worcester, Massachusetts hospitals. The model has not been externally validated on populations from other regions, demographics, or healthcare systems.

- **No uncertainty estimates:** Individual-level confidence intervals are not provided. The displayed probabilities are point estimates from the fitted model, not ranges. True individual uncertainty is higher than implied.

- **Feature exclusion:** `los` (length of hospital stay) was excluded to prevent data leakage. This may reduce predictive accuracy compared to the analysis notebook's models, which included this variable.

- **No treatment variables:** The model does not account for treatments received (medications, interventions, surgeries). Two patients with identical admission profiles may have very different outcomes depending on treatment decisions.

- **Historical cohort:** The WHAS500 dataset reflects clinical practices and patient demographics from a specific time period. Contemporary survival rates may differ due to advances in cardiac care.

- **Cross-validation is not clinical validation:** The reported C-index reflects the model's discrimination ability on held-out folds of the same dataset. It does not demonstrate clinical reliability, generalizability, or fitness for medical use.

### Technical Limitations

- **Parametric assumption (Weibull AFT):** The Weibull model assumes a specific hazard shape. If the true hazard deviates from this form, predictions may be biased, especially at extreme time horizons.

- **Proportional hazards assumption (Cox PH):** Cox PH assumes that hazard ratios remain constant over time. The Schoenfeld residuals test in the analysis notebook should be consulted to assess whether this assumption holds for each covariate.

- **Extrapolation:** Predictions beyond the observed follow-up period (2,358 days) are not supported. The 5-year (1,825-day) horizon is within the data range but has fewer observations and wider implicit uncertainty.

- **Calibration:** The models have not been formally calibrated (e.g., Hosmer-Lemeshow test, calibration plots at specific time horizons). Discrimination (C-index) alone does not guarantee that predicted probabilities match observed event rates.

---

*This calculator is labeled as an experimental research tool, not a clinically validated medical device, in compliance with responsible AI disclosure practices.*

## 11. Deployment Notes

### Local Demonstration

The calculator runs locally via Gradio's built-in web server. No cloud deployment is required for portfolio demonstration. Share your screen or record the session to showcase the tool.

### Temporary Public Link

To create a temporary public link (72 hours) for sharing:

```python
demo.launch(share=True)
```

This tunnels through Gradio's servers and requires no additional configuration. The link expires automatically.

### Permanent Deployment

For permanent hosting, deploy to [Hugging Face Spaces](https://huggingface.co/spaces):

1. Create a new Space with the Gradio SDK.
2. Upload `whas500.csv`, the `models/` folder, and a `app.py` combining the prediction engine and Gradio interface code.
3. Add a `requirements.txt` with: `lifelines`, `scikit-learn`, `plotly`, `joblib`, `pandas`, `numpy`.
4. The Space builds and deploys automatically.

---

*End of notebook.*